# **0 Import Library**

In [1]:
!pip install cartopy
!pip install rasterio
!pip install pyextremes

import ee
import geopandas as gpd
from datetime import datetime, timedelta
import pandas as pd
from pathlib import Path
import numpy as np
import rasterio
from rasterio.mask import mask
from shapely.geometry import mapping
import geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 1.9 MB/s eta 0:00:00


# **Step1 Generate CSV contain the daily rainfall data per each grid from 2024/01/01 - 2024/12/31**

In [2]:
ee.Authenticate()
ee.Initialize(project='valued-mediator-463106-q8')

In [3]:
def get_imerg_rainfall(start_time, end_time):
    imerg = ee.ImageCollection("NASA/GPM_L3/IMERG_V07").filterDate(
        start_time.strftime("%Y-%m-%d"),
        (end_time + timedelta(days=1)).strftime("%Y-%m-%d")
    ).select("precipitation")

    # Sum precipitation over the period (mm/hr to total mm)
    total_rain = imerg.sum().multiply(0.5)  # 30-min intervals to hourly
    return total_rain

In [4]:
def clip_to_czech(image, cze_shp_ee):
    return image.clip(cze_shp_ee)

In [5]:
def export_rainfall_per_grid(rainfall, sid, name, output_path, cze_shp_ee, cze_shp, cze_land, start_time, end_time):
    # Ensure output path exists
    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)

    # Export rainfall to temporary GeoTIFF
    temp_tif = output_path / f"temp_grid_{sid}.tif"
    geemap.ee_export_image(rainfall, filename=str(temp_tif),
                           region=cze_shp_ee, scale=11132)

    # Mask rainfall to land-only areas
    with rasterio.open(temp_tif) as src:
        cze_land_geom = [mapping(geom) for geom in cze_land.geometry]
        masked_rainfall, masked_transform = mask(src, cze_land_geom, crop=True, nodata=np.nan)

    rainfall_mm = masked_rainfall[0]  # 2D array of rainfall values in mm
    valid_pixels = ~np.isnan(rainfall_mm)  # Mask for non-NaN (land) pixels

    num_grids_processed = 0
    if np.any(valid_pixels):
        # Get pixel coordinates
        rows, cols = np.where(valid_pixels)
        # Convert pixel indices to geographic coordinates
        lon, lat = rasterio.transform.xy(masked_transform, rows, cols)
        # Extract rainfall values for valid pixels
        rainfall_values = rainfall_mm[valid_pixels]

        # Process each grid cell
        for lon_val, lat_val, rain_val in zip(lon, lat, rainfall_values):

            lon_formatted = f"{lon_val:.16f}"
            lat_formatted = f"{lat_val:.16f}"
            # Create grid_key with formatted coordinates
            grid_key = f"{lon_formatted}_{lat_formatted}"
            csv_filename = output_path / f"{grid_key}.csv"

            # Create DataFrame for the current period's data
            new_data = pd.DataFrame([{
                "Start_time": start_time,
                "End_time": end_time,
                "Longitude": lon_formatted,
                "Latitude": lat_formatted,
                "Rainfall_mm": rain_val

            }])

            # Check if CSV already exists
            if csv_filename.exists():
                # Read existing CSV and append new data
                existing_data = pd.read_csv(csv_filename)
                updated_data = pd.concat([existing_data, new_data], ignore_index=True)
                updated_data.to_csv(csv_filename, index=False)
                print(f"Appended data to {csv_filename}")
            else:
                # Create new CSV
                new_data.to_csv(csv_filename, index=False)
                print(f"Created new CSV for grid {grid_key}")

            num_grids_processed += 1
    # Clean up temporary GeoTIFF
    if temp_tif.exists():
        temp_tif.unlink()

    return num_grids_processed  # Return number of grids processed

It can modify the start date in     start_date = datetime(2024, 1, 1)


In [6]:
def main():
    OUTPUT_DIR = "output"  # Define output directory

    # Load Czech shapefile
    print("Loading shapefile...")
    cze_shp = gpd.read_file("ACP_adm0.shp").to_crs("EPSG:4326")
    cze_land = cze_shp.copy()  # Use the same for land masking

    # Convert shapefile to EE geometry
    cze_coords = cze_shp.geometry.unary_union.__geo_interface__["coordinates"]
    cze_shp_ee = ee.Geometry.Polygon(cze_coords)

    # Set fixed time period
    start_date = datetime(2024, 1, 1)
    end_date = datetime(2024, 12, 31)

    current_date = start_date
    while current_date <= end_date:
        day_start = current_date
        day_end = current_date + timedelta(days=1) - timedelta(seconds=1)
        day_str = day_start.strftime("%Y-%m-%d")
        sid = day_str
        name = f"DailyRainfall_{day_str}"

        print(f"Processing daily rainfall for {day_str}...")

        # Get IMERG rainfall for the day
        rainfall = get_imerg_rainfall(day_start, day_start)  # Note: end_time is day_start, but filter will go to next day
        rainfall = clip_to_czech(rainfall, cze_shp_ee)

        # Export rainfall per grid cell
        print(f"Exporting rainfall per grid for {day_str}...")
        num_grids = export_rainfall_per_grid(
            rainfall, sid, name, OUTPUT_DIR, cze_shp_ee, cze_shp, cze_land, day_start, day_end
        )
        print(f"Processed {num_grids} grid cells for {day_str}")

        current_date += timedelta(days=1)

if __name__ == "__main__":
    main()

Loading shapefile...


/tmp/ipython-input-2428436085.py:10: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  cze_coords = cze_shp.geometry.unary_union.__geo_interface__["coordinates"]


串流輸出內容已截斷至最後 5000 行。
Appended data to output/14.4500660983727514_48.0502197942429561.csv
Appended data to output/14.5500665558009370_48.0502197942429561.csv
Appended data to output/14.6500670132291226_48.0502197942429561.csv
Appended data to output/12.9500592369499756_47.9502193368147687.csv
Appended data to output/13.0500596943781595_47.9502193368147687.csv
Appended data to output/13.1500601518063451_47.9502193368147687.csv
Appended data to output/13.2500606092345308_47.9502193368147687.csv
Appended data to output/13.3500610666627146_47.9502193368147687.csv
Appended data to output/13.4500615240909003_47.9502193368147687.csv
Appended data to output/13.5500619815190859_47.9502193368147687.csv
Appended data to output/13.6500624389472698_47.9502193368147687.csv
Appended data to output/13.7500628963754554_47.9502193368147687.csv
Appended data to output/13.8500633538036411_47.9502193368147687.csv
Appended data to output/13.9500638112318249_47.9502193368147687.csv
Appended data to output/14.

In [10]:
import zipfile

def zip_csv_files(output_path):
    """
    Zip all CSV files in the output directory  .

    Parameters:
    - output_path: Path object to the directory containing CSV files
    """
    zip_filename = output_path / f"0Rainfall_GRID_Data.zip"

    # Find all CSV files in the output directory
    csv_files = list(output_path.glob("*.csv"))

    if not csv_files:
        print(f"No CSV files found in {output_path}. Skipping zipping.")
        return

    # Create a zip file and add all CSVs
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for csv_file in csv_files:
            zipf.write(csv_file, arcname=csv_file.name)
            print(f"Added {csv_file.name} to {zip_filename}")

    print(f"Created zip file: {zip_filename}")

In [11]:
OUTPUT_DIR = Path("output")

zip_csv_files(OUTPUT_DIR)

串流輸出內容已截斷至最後 5000 行。
Added 12.4500569498090492_47.3502165922456584.csv to output/0Rainfall_GRID_Data.zip
Added 15.4500706726546042_46.6502133902483607.csv to output/0Rainfall_GRID_Data.zip
Added 21.3500976609175268_50.7502321448039524.csv to output/0Rainfall_GRID_Data.zip
Added 22.1501013203430048_53.7502458676495039.csv to output/0Rainfall_GRID_Data.zip
Added 19.9500912569229314_53.7502458676495039.csv to output/0Rainfall_GRID_Data.zip
Added 22.2501017777711922_52.8502417507958384.csv to output/0Rainfall_GRID_Data.zip
Added 17.8500816509310454_52.3502394636549155.csv to output/0Rainfall_GRID_Data.zip
Added 21.5500985757738945_49.8502280279502870.csv to output/0Rainfall_GRID_Data.zip
Added 16.5500757043646374_50.7502321448039524.csv to output/0Rainfall_GRID_Data.zip
Added 15.7500720449391594_47.7502184219583938.csv to output/0Rainfall_GRID_Data.zip
Added 22.4501026926275600_54.3502486122186141.csv to output/0Rainfall_GRID_Data.zip
Added 16.8500770766491961_50.3502303150912098.csv to ou